# AWP Runtime Components Tutorial

This notebook demonstrates **all AWP runtime components** with hands-on, runnable examples.
No LLM API key is needed for most cells -- everything runs locally.

**Components covered:**
1. Tool Registry -- built-in tools, definitions, calling tools directly
2. Code Execution -- subprocess sandbox, timeouts, error handling
3. Executor Factory -- creating executors from sandbox config
4. Message Bus -- inter-agent direct messaging, broadcast, channels
5. State Persistence -- checkpoints, final state, loading
6. Security -- circuit breaker, rate limiter, access controller
7. Observability -- tracing, metrics, audit trail
8. Dynamic Tool Factory -- runtime tool creation
9. Wiring It All Together
10. Summary -- quick reference table

In [1]:
# Setup: ensure we are in the project root so all paths resolve correctly
import os, sys

PROJECT_ROOT = "/home/shumway/projects/agent-workflow-protocol"
os.chdir(PROJECT_ROOT)

# Make sure the AWP package is importable
src_path = os.path.join(PROJECT_ROOT, "reference", "python", "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"Working directory: {os.getcwd()}")
print(f"Python: {sys.version}")
print("AWP runtime components ready.")

Working directory: /home/shumway/projects/agent-workflow-protocol
Python: 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]
AWP runtime components ready.


---
## 1. Tool Registry

The `ToolRegistry` is the central hub for all AWP tools. It:
- Registers built-in tools (file, shell, arithmetic, memory, web, http)
- Discovers custom tools from a workflow's `mcp/` directory
- Provides tool definitions in OpenAI function-calling format
- Executes tools by fully-qualified name (FQN)

All tools return the standard AWP result format:
```python
{"ok": bool, "status": int, "data": Any, "error": str | None}
```

In [2]:
from awp.runtime.tools import ToolRegistry
from pathlib import Path

# Create a registry pointing at a workflow directory
reg = ToolRegistry(workflow_dir=Path("examples/workflows/01-hello-world"))

# List all built-in tool names (tool_names is a property)
print("=== All registered tools ===")
for name in reg.tool_names:
    print(f"  {name}")
print(f"\nTotal: {len(reg.tool_names)} tools")

=== All registered tools ===
  arithmetic.add
  arithmetic.divide
  arithmetic.multiply
  arithmetic.subtract
  file.list
  file.read
  file.write
  http.request
  memory.curate
  memory.read
  memory.search
  memory.write
  shell.execute
  web.search

Total: 14 tools


In [3]:
# Get tool definitions in OpenAI function-calling format
# This is what gets passed to LLMs so they know which tools are available
import json

defs = reg.get_definitions()
print(f"Total tool definitions: {len(defs)}\n")

# Show the first definition (file.read) as an example
print("=== Example: file.read definition ===")
file_read_def = [d for d in defs if d["function"]["name"] == "file.read"][0]
print(json.dumps(file_read_def, indent=2))

Total tool definitions: 14

=== Example: file.read definition ===
{
  "type": "function",
  "function": {
    "name": "file.read",
    "description": "Read file contents from disk",
    "parameters": {
      "type": "object",
      "properties": {
        "path": {
          "type": "string",
          "description": "File path to read"
        },
        "encoding": {
          "type": "string",
          "description": "File encoding",
          "default": "utf-8"
        }
      },
      "required": [
        "path"
      ]
    }
  }
}


In [4]:
# You can filter definitions by pattern (useful for agent tools_allowed)
arith_defs = reg.get_definitions(allowed=["arithmetic.*"])
print(f"Arithmetic tools only ({len(arith_defs)} tools):")
for d in arith_defs:
    print(f"  {d['function']['name']}: {d['function']['description']}")

print()
file_defs = reg.get_definitions(allowed=["file.*"])
print(f"File tools only ({len(file_defs)} tools):")
for d in file_defs:
    print(f"  {d['function']['name']}: {d['function']['description']}")

Arithmetic tools only (4 tools):
  arithmetic.add: Add two numbers
  arithmetic.subtract: Subtract two numbers
  arithmetic.multiply: Multiply two numbers
  arithmetic.divide: Divide two numbers

File tools only (3 tools):
  file.read: Read file contents from disk
  file.write: Write content to a file
  file.list: List files in a directory


### 1.1 Calling Tools Directly

Use `reg.call(tool_name, args_dict)` to invoke any tool. This is what the runtime
does when an LLM requests a tool call.

In [5]:
# --- Arithmetic tools ---
print("=== Arithmetic Tools ===")

result = reg.call("arithmetic.add", {"a": 10, "b": 20})
print(f"add(10, 20) = {result['data']['result']}  (full: {result})")

result = reg.call("arithmetic.multiply", {"a": 7, "b": 6})
print(f"multiply(7, 6) = {result['data']['result']}")

result = reg.call("arithmetic.subtract", {"a": 100, "b": 37})
print(f"subtract(100, 37) = {result['data']['result']}")

result = reg.call("arithmetic.divide", {"a": 42, "b": 7})
print(f"divide(42, 7) = {result['data']['result']}")

# Division by zero is handled gracefully
result = reg.call("arithmetic.divide", {"a": 1, "b": 0})
print(f"divide(1, 0) = ok={result['ok']}, error='{result['error']}'")

=== Arithmetic Tools ===
add(10, 20) = 30  (full: {'ok': True, 'status': 200, 'data': {'result': 30}, 'error': None, 'log': ''})
multiply(7, 6) = 42
subtract(100, 37) = 63
divide(42, 7) = 6.0
divide(1, 0) = ok=False, error='Division by zero'


In [6]:
# --- File tools ---
print("=== file.read ===")

result = reg.call("file.read", {"path": "README.md"})
print(f"ok={result['ok']}, status={result['status']}")
content = result["data"]["content"]
size = result["data"]["size"]
print(f"size={size} chars")
print(f"first 200 chars: {content[:200]}...")

=== file.read ===
ok=True, status=200
size=29744 chars
first 200 chars: <p align="center">
  <img src="assets/awp_logo.png" alt="AWP Logo" width="200" />
</p>

<h1 align="center">AWP -- Agent Workflow Protocol</h1>

<p align="center">
  <strong>The handbook: Multi-agent w...


In [7]:
# List files in a directory
print("=== file.list ===")
result = reg.call("file.list", {"path": "examples/workflows/01-hello-world"})
print(f"file.list('examples/workflows/01-hello-world'):")
print(f"  ok={result['ok']}, count={result['data']['count']}")
for f in result["data"]["files"]:
    print(f"    {f}")

print()

# Recursive listing
result = reg.call("file.list", {"path": "examples/workflows/01-hello-world", "pattern": "*.yaml", "recursive": True})
print(f"file.list (recursive, *.yaml): count={result['data']['count']}")
for f in result["data"]["files"]:
    print(f"    {f}")

=== file.list ===
file.list('examples/workflows/01-hello-world'):
  ok=True, count=1
    workflow.awp.yaml

file.list (recursive, *.yaml): count=2
    agents/greeter/agent.awp.yaml
    workflow.awp.yaml


In [8]:
# Write a file (to a temporary location)
import tempfile, os

print("=== file.write and read-back ===")
with tempfile.TemporaryDirectory() as td:
    test_file = os.path.join(td, "test_output.txt")
    result = reg.call("file.write", {"path": test_file, "content": "Hello from AWP ToolRegistry!"})
    print(f"file.write: ok={result['ok']}, path={result['data']['path']}, size={result['data']['size']}")

    # Read it back
    result = reg.call("file.read", {"path": test_file})
    print(f"file.read: content='{result['data']['content']}'")

=== file.write and read-back ===
file.write: ok=True, path=/tmp/tmplz01zo2h/test_output.txt, size=28
file.read: content='Hello from AWP ToolRegistry!'


In [9]:
# --- Shell execute ---
print("=== shell.execute ===")
result = reg.call("shell.execute", {"command": "echo 'Hello from AWP shell tool' && date"})
print(f"ok={result['ok']}, status={result['status']}")
print(f"stdout: {result['data']['stdout'].strip()}")
print(f"returncode: {result['data']['returncode']}")

=== shell.execute ===
ok=True, status=200
stdout: Hello from AWP shell tool
Sa 28. Mär 04:14:41 CET 2026
returncode: 0


In [10]:
# --- Memory tools ---
print("=== Memory Tools ===")

# Write to daily memory log
result = reg.call("memory.write", {"content": "Tutorial session started. Testing all runtime components.", "target": "daily"})
print(f"memory.write (daily): ok={result['ok']}, path={result['data'].get('path', 'n/a')}")

# Read daily memory
result = reg.call("memory.read", {"target": "daily"})
print(f"memory.read (daily): ok={result['ok']}, exists={result['data'].get('exists')}")
if result["ok"] and result["data"].get("content"):
    print(f"  Content: {result['data']['content'][:200]}")

# Search memory
result = reg.call("memory.search", {"query": "tutorial", "max_results": 5})
print(f"memory.search('tutorial'): ok={result['ok']}, matches={len(result['data'].get('results', []))}")
for r in result["data"].get("results", []):
    print(f"  [{r['source']}:{r['line']}] {r['text'][:80]}")

# List available dates
result = reg.call("memory.read", {"target": "dates"})
print(f"memory.read (dates): {result['data'].get('dates', [])}")

=== Memory Tools ===
memory.write (daily): ok=True, path=examples/workflows/01-hello-world/workspace/memory/2026-03-28.md
memory.read (daily): ok=True, exists=True
  Content: 
### 03:03:23
Tutorial session started. Testing all runtime components.

### 03:14:41
Tutorial session started. Testing all runtime components.

memory.search('tutorial'): ok=True, matches=2
  [2026-03-28.md:3] Tutorial session started. Testing all runtime components.
  [2026-03-28.md:6] Tutorial session started. Testing all runtime components.
memory.read (dates): ['2026-03-28']


In [11]:
# --- Web and HTTP tools (may return errors without API keys or network) ---
print("=== Web & HTTP Tools ===")

# web.search -- shows the tool contract even if no search engine API key is configured
result = reg.call("web.search", {"query": "AWP agent workflow protocol", "max_results": 3})
print(f"web.search: ok={result['ok']}, status={result['status']}")
if not result["ok"]:
    print(f"  (Expected without API key) error: {str(result['error'])[:120]}")
else:
    print(f"  results: {len(result['data'].get('results', []))}")

print()

# http.request -- make a simple GET request
result = reg.call("http.request", {"url": "https://httpbin.org/get", "method": "GET", "timeout": 5})
print(f"http.request(httpbin.org/get): ok={result['ok']}, status={result['status']}")
if result["ok"]:
    print(f"  Response data keys: {list(result['data'].keys())[:5]}")
else:
    print(f"  (Network may be unavailable) error: {str(result.get('error', ''))[:120]}")

=== Web & HTTP Tools ===


web.search: ok=True, status=200
  results: 3



http.request(httpbin.org/get): ok=True, status=200
  Response data keys: ['status_code', 'headers', 'body']


In [12]:
# --- Calling an unknown tool returns a clean error ---
result = reg.call("nonexistent.tool", {})
print(f"Unknown tool result: ok={result['ok']}, status={result['status']}, error='{result['error']}'")

Unknown tool result: ok=False, status=404, error='Unknown tool: nonexistent.tool'


---
## 2. Code Execution

The `CodeExecutor` runs Python code in a sandboxed subprocess with:
- Configurable timeout (capped by `max_timeout`)
- Output size limits (`max_output_bytes`)
- Clean error handling for syntax errors, runtime errors, and timeouts
- AST-based code validation (without execution)

In [13]:
from awp.runtime.code_executor import CodeExecutor

ex = CodeExecutor(max_timeout=10, max_output_bytes=100_000)

# Basic execution
print("=== Basic Execution ===")
result = ex.execute("print(2 + 2)")
print(f"ok={result['ok']}, status={result['status']}")
print(f"stdout='{result['data']['stdout'].strip()}'")
print(f"stderr='{result['data']['stderr']}'")
print(f"returncode={result['data']['returncode']}")
print(f"error={result['error']}")

=== Basic Execution ===
ok=True, status=200
stdout='4'
stderr=''
returncode=0
error=None


In [14]:
# Multi-line code with imports
print("=== Multi-line Code ===")
code = """
import math
import json

data = {
    "pi": round(math.pi, 6),
    "e": round(math.e, 6),
    "sqrt2": round(math.sqrt(2), 6),
    "factorial_10": math.factorial(10),
}
print(json.dumps(data, indent=2))
"""
result = ex.execute(code)
print(f"ok={result['ok']}")
print(f"Output:\n{result['data']['stdout']}")

=== Multi-line Code ===
ok=True
Output:
{
  "pi": 3.141593,
  "e": 2.718282,
  "sqrt2": 1.414214,
  "factorial_10": 3628800
}



In [15]:
# Syntax error handling
print("=== Syntax Error ===")
result = ex.execute("def broken(:\n  pass")
print(f"ok={result['ok']}, status={result['status']}")
print(f"error: {result.get('error', '')[:200]}")

=== Syntax Error ===
ok=False, status=500
error:   File "/tmp/tmpu6fubo6k.py", line 1
    def broken(:
               ^
SyntaxError: invalid syntax



In [16]:
# Runtime error handling
print("=== Runtime Error ===")
result = ex.execute("x = 1 / 0")
print(f"ok={result['ok']}, status={result['status']}")
print(f"error: {result.get('error', '')[:200]}")

=== Runtime Error ===


ok=False, status=500
error: Traceback (most recent call last):
  File "/tmp/tmpmodf_hxj.py", line 1, in <module>
    x = 1 / 0
        ~~^~~
ZeroDivisionError: division by zero



In [17]:
# Timeout handling
print("=== Timeout Handling ===")
short_executor = CodeExecutor(max_timeout=2, max_output_bytes=100_000)
result = short_executor.execute("import time; time.sleep(10); print('done')")
print(f"ok={result['ok']}, status={result['status']}")
print(f"error: {result['error']}")

=== Timeout Handling ===


ok=False, status=408
error: Code execution timed out after 2s


In [18]:
# Output size limits
print("=== Output Size Limits ===")
small_executor = CodeExecutor(max_timeout=5, max_output_bytes=50)
result = small_executor.execute("print('A' * 1000)")
print(f"ok={result['ok']}")
stdout = result['data']['stdout']
print(f"stdout length: {len(stdout)} (capped at 50 bytes)")
print(f"stdout: '{stdout[:60]}'")

=== Output Size Limits ===
ok=True
stdout length: 50 (capped at 50 bytes)
stdout: 'AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA'


In [19]:
# Code validation (AST-based, no execution)
print("=== Code Validation (AST only, no execution) ===")

valid = ex.validate_code("x = 42\nprint(x)")
print(f"Valid code:   ok={valid['ok']}, data={valid['data']}")

invalid = ex.validate_code("def broken(:")
print(f"Invalid code: ok={invalid['ok']}, error='{invalid['error']}'")

=== Code Validation (AST only, no execution) ===
Valid code:   ok=True, data={'valid': True}
Invalid code: ok=False, error='Syntax error: invalid syntax (<unknown>, line 1)'


---
## 3. Executor Factory

The `create_executor()` factory reads a `SandboxConfig` and returns the appropriate
executor instance. Supported types:
- `subprocess` (default) -- runs in a child process
- `docker` -- runs in a Docker container
- `venv` -- runs in a virtual environment
- `none` -- falls back to subprocess

In [20]:
from awp.runtime.executor_factory import create_executor
from awp.models.capabilities import SandboxConfig

# Default: subprocess executor
print("=== Default (subprocess) ===")
config = SandboxConfig(enabled=True, type="subprocess", timeout=10)
executor = create_executor(config)
print(f"Executor type: {type(executor).__name__}")
result = executor.execute("import sys; print(f'Python {sys.version_info.major}.{sys.version_info.minor}')")
print(f"ok={result['ok']}, stdout='{result['data']['stdout'].strip()}'")

print()

# None config defaults to subprocess
print("=== None config (defaults) ===")
executor2 = create_executor(None)
print(f"Executor type: {type(executor2).__name__}")
result = executor2.execute("print('hello from default executor')")
print(f"ok={result['ok']}, stdout='{result['data']['stdout'].strip()}'")

=== Default (subprocess) ===
Executor type: CodeExecutor
ok=True, stdout='Python 3.12'

=== None config (defaults) ===
Executor type: CodeExecutor
ok=True, stdout='hello from default executor'


In [21]:
# Show all SandboxConfig fields and their defaults
print("=== SandboxConfig Fields ===")
config = SandboxConfig()
for field_name, field_info in SandboxConfig.model_fields.items():
    value = getattr(config, field_name)
    print(f"  {field_name}: {value!r}")

=== SandboxConfig Fields ===
  enabled: False
  type: 'subprocess'
  runtime: 'python'
  timeout: 30
  max_memory_mb: 256
  max_cpu_seconds: 30
  max_output_bytes: 1048576
  allowed_modules: []
  network_access: False
  packages: []
  image: 'awp-sandbox-python'
  pip_install: False


In [22]:
# Custom config with tight limits
print("=== Custom Config (tight limits) ===")
tight_config = SandboxConfig(
    enabled=True,
    type="subprocess",
    timeout=3,
    max_output_bytes=100,
    max_memory_mb=64,
)
tight_executor = create_executor(tight_config)

result = tight_executor.execute("print('A' * 200)")
print(f"ok={result['ok']}")
print(f"stdout length: {len(result['data']['stdout'])} (capped at 100 bytes)")

=== Custom Config (tight limits) ===
ok=True
stdout length: 100 (capped at 100 bytes)


---
## 4. Message Bus

The `MessageBus` provides in-memory inter-agent communication with:
- **Direct messaging** -- point-to-point between named agents
- **Broadcast** -- to all agents (recipient = `*`)
- **Channels** -- logical grouping of messages
- **Filtering** -- by channel, sender, with limits

Production deployments can swap in Redis, NATS, Kafka, etc.

In [23]:
from awp.runtime.message_bus import MessageBus

bus = MessageBus()

# Direct message: coordinator -> specialist
print("=== Direct Messaging ===")
msg_id1 = bus.send(
    from_agent="coordinator",
    to_agent="specialist",
    content={"task": "analyze", "data": [1, 2, 3]},
    channel="tasks",
)
print(f"Message 1 (coordinator -> specialist): {msg_id1[:12]}...")

# Another direct message to a different agent
msg_id2 = bus.send(
    from_agent="coordinator",
    to_agent="writer",
    content={"task": "write_report", "format": "markdown"},
    channel="tasks",
)
print(f"Message 2 (coordinator -> writer): {msg_id2[:12]}...")

# Specialist responds
msg_id3 = bus.send(
    from_agent="specialist",
    to_agent="coordinator",
    content={"status": "done", "result": {"mean": 2.0}},
    channel="results",
    msg_type="response",
)
print(f"Message 3 (specialist -> coordinator): {msg_id3[:12]}...")

=== Direct Messaging ===
Message 1 (coordinator -> specialist): c2f85dbe-a50...
Message 2 (coordinator -> writer): 85246e9f-bc5...
Message 3 (specialist -> coordinator): 476f6dce-b07...


In [24]:
# Broadcast: specialist announces status to all agents
print("=== Broadcast ===")
broadcast_id = bus.broadcast(
    from_agent="specialist",
    content={"status": "analysis_complete", "confidence": 0.95},
    channel="status",
)
print(f"Broadcast sent (specialist -> *): {broadcast_id[:12]}...")

=== Broadcast ===
Broadcast sent (specialist -> *): cc052792-8f9...


In [25]:
# List messages for an agent (includes direct + broadcasts from others)
print("=== Messages for 'specialist' ===")
msgs = bus.list_messages(agent_id="specialist")
for m in msgs:
    print(f"  [{m['channel']}] from={m['from']} -> to={m['to']}: {m['content']}")

print()
print("=== Messages for 'coordinator' ===")
msgs = bus.list_messages(agent_id="coordinator")
for m in msgs:
    print(f"  [{m['channel']}] from={m['from']} -> to={m['to']}: {m['content']}")

=== Messages for 'specialist' ===
  [tasks] from=coordinator -> to=specialist: {'task': 'analyze', 'data': [1, 2, 3]}

=== Messages for 'coordinator' ===
  [status] from=specialist -> to=*: {'status': 'analysis_complete', 'confidence': 0.95}
  [results] from=specialist -> to=coordinator: {'status': 'done', 'result': {'mean': 2.0}}


In [26]:
# Get all messages on a specific channel
print("=== Channel: 'tasks' ===")
task_msgs = bus.get_channel_messages("tasks")
for m in task_msgs:
    print(f"  {m['from']} -> {m['to']}: {m['content']}")

print()
print("=== Channel: 'status' ===")
status_msgs = bus.get_channel_messages("status")
for m in status_msgs:
    print(f"  {m['from']} -> {m['to']}: {m['content']}")

print()
print("=== Filtering: coordinator's messages from specialist only ===")
filtered = bus.list_messages(agent_id="coordinator", from_agent="specialist")
for m in filtered:
    print(f"  [{m['channel']}] {m['content']}")

print()
print("=== Clear and verify ===")
bus.clear()
print(f"Messages after clear: {len(bus.get_channel_messages('tasks'))}")

=== Channel: 'tasks' ===
  coordinator -> specialist: {'task': 'analyze', 'data': [1, 2, 3]}
  coordinator -> writer: {'task': 'write_report', 'format': 'markdown'}

=== Channel: 'status' ===
  specialist -> *: {'status': 'analysis_complete', 'confidence': 0.95}

=== Filtering: coordinator's messages from specialist only ===
  [status] {'status': 'analysis_complete', 'confidence': 0.95}
  [results] {'status': 'done', 'result': {'mean': 2.0}}

=== Clear and verify ===
Messages after clear: 0


---
## 5. State Persistence

The `StatePersistence` class saves JSON state snapshots to disk:
- **Per-agent checkpoints** -- saved after each agent completes
- **Final state** -- the complete workflow result
- **Load/restore** -- resume from checkpoints

In [27]:
from awp.runtime.state_persistence import StatePersistence
from pathlib import Path
import tempfile
import json

with tempfile.TemporaryDirectory() as td:
    sp = StatePersistence(output_dir=Path(td))

    # Save per-agent checkpoints
    print("=== Saving Checkpoints ===")
    path1 = sp.save_checkpoint("researcher", {
        "confidence": 0.92,
        "findings": ["result_a", "result_b"],
        "tokens_used": 1500,
    })
    print(f"Saved researcher checkpoint: {Path(path1).name}")

    path2 = sp.save_checkpoint("writer", {
        "confidence": 0.88,
        "report": "Analysis shows positive trend...",
        "word_count": 250,
    })
    print(f"Saved writer checkpoint: {Path(path2).name}")

    # Save final workflow state
    final_path = sp.save_final({
        "workflow_status": "completed",
        "agents": {
            "researcher": {"confidence": 0.92},
            "writer": {"confidence": 0.88},
        },
        "total_tokens": 3200,
    })
    print(f"Saved final state: {Path(final_path).name}")

    # Load checkpoints back
    print("\n=== Loading Checkpoints ===")
    researcher_state = sp.load_checkpoint("researcher")
    print(f"Researcher: {json.dumps(researcher_state, indent=2)}")

    writer_state = sp.load_checkpoint("writer")
    print(f"Writer: {json.dumps(writer_state, indent=2)}")

    # Load final state
    final = sp.load_final()
    print(f"\nFinal: {json.dumps(final, indent=2)}")

    # Non-existent checkpoint returns None
    missing = sp.load_checkpoint("nonexistent_agent")
    print(f"\nMissing checkpoint: {missing}")

=== Saving Checkpoints ===
Saved researcher checkpoint: researcher.json
Saved writer checkpoint: writer.json
Saved final state: final.json

=== Loading Checkpoints ===
Researcher: {
  "confidence": 0.92,
  "findings": [
    "result_a",
    "result_b"
  ],
  "tokens_used": 1500
}
Writer: {
  "confidence": 0.88,
  "report": "Analysis shows positive trend...",
  "word_count": 250
}

Final: {
  "workflow_status": "completed",
  "agents": {
    "researcher": {
      "confidence": 0.92
    },
    "writer": {
      "confidence": 0.88
    }
  },
  "total_tokens": 3200
}

Missing checkpoint: None


---
## 6. Security Components

AWP provides three security primitives:
- **CircuitBreaker** -- stops calling a failing service after N failures
- **RateLimiter** -- sliding window rate limiting per agent
- **AccessController** -- tool-level access policies per agent

These compose into a `SecurityContext` that the runtime enforces automatically.

In [28]:
from awp.runtime.security import CircuitBreaker, RateLimiter, AccessController, SecurityContext

# --- Circuit Breaker ---
print("=== Circuit Breaker ===")
cb = CircuitBreaker(failure_threshold=3, reset_timeout=2.0, half_open_max_calls=1)

print(f"Initial state: {cb.state}")
print(f"Call allowed: {cb.check()}")

# Simulate failures to trip the breaker
print("\nSimulating 3 consecutive failures...")
for i in range(3):
    cb.record_failure()
    print(f"  Failure {i+1}: state={cb.state}, call_allowed={cb.check()}")

print(f"\nCircuit is OPEN -- all calls are blocked.")

Circuit breaker: closed → open (failures=3)


=== Circuit Breaker ===
Initial state: closed
Call allowed: True

Simulating 3 consecutive failures...
  Failure 1: state=closed, call_allowed=True
  Failure 2: state=closed, call_allowed=True
  Failure 3: state=open, call_allowed=False

Circuit is OPEN -- all calls are blocked.


In [29]:
import time

# Demonstrate full lifecycle: closed -> open -> half_open -> closed
print("=== Circuit Breaker: Full Lifecycle ===")
cb2 = CircuitBreaker(failure_threshold=2, reset_timeout=1.0, half_open_max_calls=1)

# Trip the breaker
cb2.record_failure()
cb2.record_failure()
print(f"After 2 failures: state={cb2.state}")

# Wait for reset timeout -> transitions to half_open
print("Waiting 1.1s for reset timeout...")
time.sleep(1.1)
print(f"After timeout: state={cb2.state} (allows 1 test call)")
print(f"Test call allowed: {cb2.check()}")

# A success in half_open closes the circuit
cb2.record_success()
print(f"After success: state={cb2.state} (fully recovered!)")
print(f"Calls allowed again: {cb2.check()}")

Circuit breaker: closed → open (failures=2)


=== Circuit Breaker: Full Lifecycle ===
After 2 failures: state=open
Waiting 1.1s for reset timeout...


After timeout: state=half_open (allows 1 test call)
Test call allowed: True
After success: state=closed (fully recovered!)
Calls allowed again: True


In [30]:
# --- Rate Limiter ---
print("=== Rate Limiter ===")
rl = RateLimiter(max_calls_per_minute=5, per_agent=True)

# Simulate calls from two agents
print("agent_a (limit=5/min):")
for i in range(7):
    allowed = rl.check("agent_a")
    if allowed:
        rl.record("agent_a")
    print(f"  call {i+1}: allowed={allowed}")

print()

# agent_b has its own independent window
print("agent_b (separate quota):")
for i in range(3):
    allowed = rl.check("agent_b")
    if allowed:
        rl.record("agent_b")
    print(f"  call {i+1}: allowed={allowed}")

print("\nagent_a is rate-limited, but agent_b still has quota.")

=== Rate Limiter ===
agent_a (limit=5/min):
  call 1: allowed=True
  call 2: allowed=True
  call 3: allowed=True
  call 4: allowed=True
  call 5: allowed=True
  call 6: allowed=False
  call 7: allowed=False

agent_b (separate quota):
  call 1: allowed=True
  call 2: allowed=True
  call 3: allowed=True

agent_a is rate-limited, but agent_b still has quota.


In [31]:
# --- Access Controller ---
print("=== Access Controller ===")
ac = AccessController(
    default_policy="allow",
    rules=[
        {"agent": "junior_agent", "deny_tools": ["shell.execute", "file.write"]},
        {"agent": "read_only_agent", "deny_tools": ["shell.execute", "file.write", "http.request"]},
    ],
)

# Senior agent has no restrictions
print("senior_agent (no restrictions):")
for tool in ["file.read", "file.write", "shell.execute"]:
    print(f"  {tool}: allowed={ac.is_allowed('senior_agent', tool)}")

print()
print("junior_agent (no shell/write):")
for tool in ["file.read", "file.write", "shell.execute", "arithmetic.add"]:
    print(f"  {tool}: allowed={ac.is_allowed('junior_agent', tool)}")

print()
print("read_only_agent (heavily restricted):")
for tool in ["file.read", "file.write", "shell.execute", "http.request", "memory.read"]:
    print(f"  {tool}: allowed={ac.is_allowed('read_only_agent', tool)}")

Access denied: agent 'junior_agent' cannot use tool 'file.write'


Access denied: agent 'junior_agent' cannot use tool 'shell.execute'


Access denied: agent 'read_only_agent' cannot use tool 'file.write'


Access denied: agent 'read_only_agent' cannot use tool 'shell.execute'


Access denied: agent 'read_only_agent' cannot use tool 'http.request'


=== Access Controller ===
senior_agent (no restrictions):
  file.read: allowed=True
  file.write: allowed=True
  shell.execute: allowed=True

junior_agent (no shell/write):
  file.read: allowed=True
  file.write: allowed=False
  shell.execute: allowed=False
  arithmetic.add: allowed=True

read_only_agent (heavily restricted):
  file.read: allowed=True
  file.write: allowed=False
  shell.execute: allowed=False
  http.request: allowed=False
  memory.read: allowed=True


In [32]:
# --- SecurityContext: composing all three ---
print("=== SecurityContext (composed) ===")
ctx = SecurityContext(
    circuit_breaker=CircuitBreaker(failure_threshold=5),
    rate_limiter=RateLimiter(max_calls_per_minute=100),
    access_controller=AccessController(
        default_policy="allow",
        rules=[{"agent": "restricted", "deny_tools": ["shell.execute"]}],
    ),
)

print(f"Circuit breaker state: {ctx.circuit_breaker.state}")
print(f"Rate limit check (agent_x): {ctx.rate_limiter.check('agent_x')}")
print(f"Access (restricted, shell.execute): {ctx.access_controller.is_allowed('restricted', 'shell.execute')}")
print(f"Access (restricted, file.read): {ctx.access_controller.is_allowed('restricted', 'file.read')}")

Access denied: agent 'restricted' cannot use tool 'shell.execute'


=== SecurityContext (composed) ===
Circuit breaker state: closed
Rate limit check (agent_x): True
Access (restricted, shell.execute): False
Access (restricted, file.read): True


---
## 7. Observability

AWP provides file-based observability with three components:
- **Tracer** -- span-based tracing (JSONL output)
- **MetricsCollector** -- counters and histograms (JSON output)
- **AuditTrail** -- hash-chain integrity audit log (JSONL output)

These compose into an `ObservabilityContext` that the runtime manages.

In [33]:
from awp.runtime.observability import Tracer, MetricsCollector, AuditTrail, ObservabilityContext
import tempfile
import time

with tempfile.TemporaryDirectory() as td:
    trace_dir = Path(td) / "traces"

    # --- Tracer ---
    print("=== Tracer ===")
    tracer = Tracer(output_dir=trace_dir, run_id="tutorial-run-001")

    # Start a parent span for the whole workflow
    workflow_span = tracer.start_span("workflow.execute", attributes={"workflow": "tutorial"})

    # Start a child span for an agent
    agent_span = tracer.start_span(
        "agent.researcher",
        parent_id=workflow_span,
        attributes={"model": "gpt-4o", "max_tokens": 4000},
    )
    time.sleep(0.1)  # Simulate work
    tracer.end_span(agent_span, status="ok", attributes={"tokens_used": 1234})

    # Another child span for a tool call
    tool_span = tracer.start_span(
        "tool.file.read",
        parent_id=workflow_span,
        attributes={"tool": "file.read", "path": "data.csv"},
    )
    time.sleep(0.05)  # Simulate I/O
    tracer.end_span(tool_span, status="ok")

    # End the workflow span
    tracer.end_span(workflow_span, status="ok", attributes={"total_agents": 2})

    # Flush to disk
    trace_path = tracer.flush()
    print(f"Traces written to: {trace_path.name}")

    # Read and display the traces
    trace_content = trace_path.read_text()
    for line in trace_content.strip().split("\n"):
        span = json.loads(line)
        parent = span.get('parent_id', '')[:8] if span.get('parent_id') else 'root'
        print(f"  [{span['name']}] duration={span['duration_ms']}ms status={span['status']} parent={parent}")

=== Tracer ===
Traces written to: tutorial-run-001.jsonl
  [agent.researcher] duration=100.08ms status=ok parent=4a197f4d
  [tool.file.read] duration=50.07ms status=ok parent=4a197f4d
  [workflow.execute] duration=150.27ms status=ok parent=root


In [34]:
with tempfile.TemporaryDirectory() as td:
    metrics_dir = Path(td) / "metrics"

    # --- MetricsCollector ---
    print("=== MetricsCollector ===")
    metrics = MetricsCollector(output_dir=metrics_dir, run_id="tutorial-run-001")

    # Count tool calls
    metrics.increment("tool_calls", labels={"tool": "file.read"})
    metrics.increment("tool_calls", labels={"tool": "file.read"})
    metrics.increment("tool_calls", labels={"tool": "arithmetic.add"})
    metrics.increment("tool_calls", labels={"tool": "shell.execute"})

    # Count tokens
    metrics.increment("tokens_used", value=1500, labels={"agent": "researcher"})
    metrics.increment("tokens_used", value=800, labels={"agent": "writer"})

    # Record latency histograms
    metrics.histogram("agent_latency_ms", 150.5, labels={"agent": "researcher"})
    metrics.histogram("agent_latency_ms", 230.1, labels={"agent": "researcher"})
    metrics.histogram("agent_latency_ms", 95.3, labels={"agent": "writer"})
    metrics.histogram("tool_latency_ms", 12.5, labels={"tool": "file.read"})
    metrics.histogram("tool_latency_ms", 8.2, labels={"tool": "file.read"})

    # Flush to disk
    metrics_path = metrics.flush()
    print(f"Metrics written to: {metrics_path.name}")

    # Display metrics
    metrics_data = json.loads(metrics_path.read_text())
    print(f"\nCounters:")
    for name, value in metrics_data["counters"].items():
        print(f"  {name}: {value}")
    print(f"\nHistograms:")
    for name, hist in metrics_data["histograms"].items():
        print(f"  {name}: count={hist['count']}, min={hist['min']}, max={hist['max']}, sum={hist['sum']:.1f}")

=== MetricsCollector ===
Metrics written to: tutorial-run-001.json

Counters:
  tool_calls{tool=file.read}: 2.0
  tool_calls{tool=arithmetic.add}: 1.0
  tool_calls{tool=shell.execute}: 1.0
  tokens_used{agent=researcher}: 1500.0
  tokens_used{agent=writer}: 800.0

Histograms:
  agent_latency_ms{agent=researcher}: count=2, min=150.5, max=230.1, sum=380.6
  agent_latency_ms{agent=writer}: count=1, min=95.3, max=95.3, sum=95.3
  tool_latency_ms{tool=file.read}: count=2, min=8.2, max=12.5, sum=20.7


In [35]:
with tempfile.TemporaryDirectory() as td:
    audit_dir = Path(td) / "audit"

    # --- AuditTrail ---
    print("=== AuditTrail ===")
    audit = AuditTrail(output_dir=audit_dir, run_id="tutorial-run-001")

    # Record events with hash chain integrity
    e1 = audit.record("workflow.start", details={"workflow": "tutorial"})
    print(f"Event 1: seq={e1['seq']}, type={e1['event_type']}, hash={e1['hash'][:16]}...")

    e2 = audit.record("agent.start", agent_id="researcher", details={"model": "gpt-4o"})
    print(f"Event 2: seq={e2['seq']}, type={e2['event_type']}, hash={e2['hash'][:16]}...")

    e3 = audit.record("tool.call", agent_id="researcher", details={"tool": "file.read", "path": "data.csv"})
    print(f"Event 3: seq={e3['seq']}, type={e3['event_type']}, hash={e3['hash'][:16]}...")

    e4 = audit.record("agent.complete", agent_id="researcher", details={"confidence": 0.92})
    print(f"Event 4: seq={e4['seq']}, type={e4['event_type']}, hash={e4['hash'][:16]}...")

    e5 = audit.record("workflow.complete", details={"status": "success"})
    print(f"Event 5: seq={e5['seq']}, type={e5['event_type']}, hash={e5['hash'][:16]}...")

    # Verify hash chain integrity
    entries = [e1, e2, e3, e4, e5]
    is_valid = AuditTrail.verify_chain(entries)
    print(f"\nHash chain integrity: {'VALID' if is_valid else 'INVALID'}")

    # Tamper with an entry to show detection
    tampered = [dict(e) for e in entries]
    tampered[2]["details"] = {"tool": "shell.execute", "command": "rm -rf /"}  # tampered!
    is_valid_tampered = AuditTrail.verify_chain(tampered)
    print(f"Tampered chain integrity: {'VALID' if is_valid_tampered else 'INVALID (tampering detected!)'}")

    # Flush to disk
    audit_path = audit.flush()
    print(f"\nAudit trail written to: {audit_path.name}")

=== AuditTrail ===
Event 1: seq=1, type=workflow.start, hash=d29ad7efd580dd48...
Event 2: seq=2, type=agent.start, hash=e4ca908280f85c45...
Event 3: seq=3, type=tool.call, hash=aeedee6615b009da...
Event 4: seq=4, type=agent.complete, hash=aeffc3d5d9f80374...
Event 5: seq=5, type=workflow.complete, hash=0844fe541e419d19...

Hash chain integrity: VALID
Tampered chain integrity: INVALID (tampering detected!)

Audit trail written to: tutorial-run-001.jsonl


In [36]:
# --- ObservabilityContext: composing all three ---
print("=== ObservabilityContext (composed) ===")

with tempfile.TemporaryDirectory() as td:
    base = Path(td)
    obs = ObservabilityContext(
        tracer=Tracer(output_dir=base / "traces", run_id="demo-001"),
        metrics=MetricsCollector(output_dir=base / "metrics", run_id="demo-001"),
        audit=AuditTrail(output_dir=base / "audit", run_id="demo-001"),
    )

    # Use all three together in a simulated workflow step
    span = obs.tracer.start_span("demo.operation")
    obs.metrics.increment("demo_operations")
    obs.audit.record("demo.start")

    time.sleep(0.05)

    obs.tracer.end_span(span, status="ok")
    obs.metrics.histogram("demo_latency_ms", 50.0)
    obs.audit.record("demo.complete", details={"status": "ok"})

    # Flush all at once
    obs.flush_all()

    # Show what was written
    print("Files written:")
    for subdir in ["traces", "metrics", "audit"]:
        files = list((base / subdir).glob("*"))
        for f in files:
            print(f"  {subdir}/{f.name} ({f.stat().st_size} bytes)")

    print("\nAll observability data flushed successfully.")

=== ObservabilityContext (composed) ===


Files written:
  traces/demo-001.jsonl (231 bytes)
  metrics/demo-001.json (293 bytes)
  audit/demo-001.jsonl (569 bytes)

All observability data flushed successfully.


---
## 8. Dynamic Tool Factory

The `DynamicToolFactory` lets agents create new tools at runtime:
- Validates code via AST (no dangerous imports)
- Enforces namespace rules (reserved namespaces are protected)
- Per-agent and global tool limits
- Sandboxed execution via the CodeExecutor
- Optional persistence to disk

In [37]:
from awp.runtime.dynamic_tool_factory import DynamicToolFactory, RESERVED_NAMESPACES, IMPORT_POLICIES
from awp.runtime.tools import ToolRegistry
from awp.runtime.code_executor import CodeExecutor

# Show reserved namespaces (agents cannot create tools in these)
print("=== Reserved Namespaces ===")
print(f"  {sorted(RESERVED_NAMESPACES)}")

print("\n=== Import Policies by Sandbox Type ===")
for sandbox_type, denied in IMPORT_POLICIES.items():
    print(f"  {sandbox_type}: {len(denied)} denied modules")
    if denied:
        shown = sorted(denied)[:8]
        suffix = f'... (+{len(denied)-8} more)' if len(denied) > 8 else ''
        print(f"    {shown}{suffix}")

=== Reserved Namespaces ===
  ['agent', 'arithmetic', 'code', 'doc', 'file', 'http', 'matplot', 'memory', 'numpy', 'pandas', 'shell', 'sklearn', 'tools', 'web']

=== Import Policies by Sandbox Type ===
  subprocess: 18 denied modules
    ['asyncio', 'ctypes', 'glob', 'http', 'httpx', 'importlib', 'multiprocessing', 'os']... (+10 more)
  venv: 9 denied modules
    ['ctypes', 'importlib', 'multiprocessing', 'os', 'shutil', 'signal', 'socket', 'subprocess']... (+1 more)
  docker: 2 denied modules
    ['ctypes', 'signal']
  none: 0 denied modules


In [38]:
# Create a factory with dynamic tools enabled
dyn_reg = ToolRegistry()
dyn_executor = CodeExecutor(max_timeout=10)

factory = DynamicToolFactory(
    registry=dyn_reg,
    code_executor=dyn_executor,
    config={"enabled": True, "persist": False, "max_total": 20, "allowed_namespaces": ["dynamic", "custom"]},
)

print(f"Factory enabled: {factory.enabled}")
print(f"Initial tools in registry: {len(dyn_reg.tool_names)}")

Factory enabled: True
Initial tools in registry: 14


In [39]:
# Create a simple dynamic tool
print("=== Creating Dynamic Tool: fibonacci ===")
result = factory.create_tool(
    name="dynamic.fibonacci",
    description="Calculate the Nth Fibonacci number",
    parameters={
        "type": "object",
        "properties": {
            "n": {"type": "integer", "description": "Position in Fibonacci sequence"},
        },
        "required": ["n"],
    },
    code="""def handler(*, n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return {"ok": True, "result": a, "n": n}
""",
    creator_agent="math_agent",
    allowed_namespace="dynamic",
)
print(f"Create result: ok={result['ok']}")
print(f"'dynamic.fibonacci' in tool_names: {'dynamic.fibonacci' in dyn_reg.tool_names}")
print(f"Total tools now: {len(dyn_reg.tool_names)}")

=== Creating Dynamic Tool: fibonacci ===
Create result: ok=True
'dynamic.fibonacci' in tool_names: True
Total tools now: 15


In [40]:
# Call the dynamic tool through the registry
print("=== Calling dynamic.fibonacci ===")
for n in [0, 1, 5, 10, 20]:
    result = dyn_reg.call("dynamic.fibonacci", {"n": n})
    if isinstance(result, dict) and "result" in result:
        print(f"  fibonacci({n}) = {result['result']}")
    else:
        print(f"  fibonacci({n}) = {result}")

=== Calling dynamic.fibonacci ===
  fibonacci(0) = 0
  fibonacci(1) = 1
  fibonacci(5) = 5
  fibonacci(10) = 55
  fibonacci(20) = 6765


In [41]:
# Create another tool in the 'custom' namespace
print("=== Creating Dynamic Tool: custom.uppercase ===")
result = factory.create_tool(
    name="custom.uppercase",
    description="Convert text to uppercase",
    parameters={
        "type": "object",
        "properties": {
            "text": {"type": "string", "description": "Text to convert"},
        },
        "required": ["text"],
    },
    code="""def handler(*, text):
    return {"ok": True, "result": text.upper()}
""",
    creator_agent="text_agent",
    allowed_namespace="custom",
)
print(f"Create result: ok={result['ok']}")

# Call it
result = dyn_reg.call("custom.uppercase", {"text": "hello awp world"})
if isinstance(result, dict) and "result" in result:
    print(f"custom.uppercase('hello awp world') = '{result['result']}'")
else:
    print(f"custom.uppercase result: {result}")

=== Creating Dynamic Tool: custom.uppercase ===
Create result: ok=True
custom.uppercase('hello awp world') = 'HELLO AWP WORLD'


In [42]:
# List all dynamic tools
print("=== List Dynamic Tools ===")
listing = factory.list_tools()
print(f"Total dynamic tools: {listing['data']['count']}")
for tool in listing["data"]["tools"]:
    print(f"  {tool['name']}: {tool['description']} (by {tool['creator']})")

=== List Dynamic Tools ===
Total dynamic tools: 2
  dynamic.fibonacci: Calculate the Nth Fibonacci number (by math_agent)
  custom.uppercase: Convert text to uppercase (by text_agent)


In [43]:
# --- Validation: code with forbidden imports is rejected ---
print("=== Validation: Forbidden Imports ===")
bad_result = factory.create_tool(
    name="dynamic.bad_tool",
    description="Tool with forbidden import",
    parameters={"type": "object", "properties": {}},
    code="""import os
def handler():
    return {"files": os.listdir("/")}
""",
    creator_agent="evil_agent",
    allowed_namespace="dynamic",
)
print(f"ok={bad_result['ok']}, error={bad_result['error']}")

print()

# --- Validation: reserved namespace is rejected ---
print("=== Validation: Reserved Namespace ===")
reserved_result = factory.create_tool(
    name="file.malicious",
    description="Trying to hijack file namespace",
    parameters={"type": "object", "properties": {}},
    code="""def handler():
    return {"ok": True}
""",
    creator_agent="evil_agent",
    allowed_namespace="file",
)
print(f"ok={reserved_result['ok']}, error={reserved_result['error']}")

print()

# --- Validation: missing handler function ---
print("=== Validation: Missing Handler ===")
no_handler = factory.create_tool(
    name="dynamic.no_handler",
    description="Code without handler function",
    parameters={"type": "object", "properties": {}},
    code="""def my_func():
    return 42
""",
    creator_agent="forgetful_agent",
    allowed_namespace="dynamic",
)
print(f"ok={no_handler['ok']}, error={no_handler['error']}")

=== Validation: Forbidden Imports ===
ok=False, error=Import of 'os' is not allowed in dynamic tool code (sandbox type: subprocess)

=== Validation: Reserved Namespace ===
ok=False, error=Namespace 'file' is reserved

=== Validation: Missing Handler ===
ok=False, error=Tool code must contain a 'def handler(*, ...)' function


In [44]:
# Remove a dynamic tool
print("=== Remove Dynamic Tool ===")
print(f"Before removal: {factory.list_tools()['data']['count']} dynamic tools")

remove_result = factory.remove_tool("dynamic.fibonacci", requester_agent="math_agent")
print(f"Remove result: ok={remove_result['ok']}")
print(f"After removal: {factory.list_tools()['data']['count']} dynamic tools")

# Only the creator can remove a tool
wrong_agent = factory.remove_tool("custom.uppercase", requester_agent="wrong_agent")
print(f"\nWrong agent removal: ok={wrong_agent['ok']}, error={wrong_agent['error']}")

# Cleanup all dynamic tools
factory.cleanup()
print(f"\nAfter cleanup: {factory.list_tools()['data']['count']} dynamic tools")

=== Remove Dynamic Tool ===
Before removal: 2 dynamic tools
Remove result: ok=True
After removal: 1 dynamic tools

Wrong agent removal: ok=False, error=Agent 'wrong_agent' cannot remove tool created by 'text_agent'

After cleanup: 0 dynamic tools


---
## 9. Wiring It All Together

In production, the `ToolRegistry` integrates with MessageBus, CodeExecutor,
and SecurityContext via setter methods. Here is a complete wiring example:

In [45]:
from awp.runtime.tools import ToolRegistry
from awp.runtime.code_executor import CodeExecutor
from awp.runtime.message_bus import MessageBus
from awp.runtime.security import SecurityContext, CircuitBreaker, RateLimiter, AccessController

# Create all components
registry = ToolRegistry()
code_exec = CodeExecutor(max_timeout=10)
msg_bus = MessageBus()
sec_ctx = SecurityContext(
    circuit_breaker=CircuitBreaker(),
    rate_limiter=RateLimiter(),
    access_controller=AccessController(default_policy="allow"),
)

# Wire them into the registry
print(f"Tools BEFORE wiring: {len(registry.tool_names)}")
registry.set_code_executor(code_exec)    # Adds code.execute tool
registry.set_message_bus(msg_bus)         # Adds agent.send_message + agent.list_messages
registry.set_security_context(sec_ctx)   # Enables access control on tool calls
print(f"Tools AFTER wiring:  {len(registry.tool_names)}")

print("\n=== All tools after full wiring ===")
for name in registry.tool_names:
    print(f"  {name}")

Tools BEFORE wiring: 14
Tools AFTER wiring:  17

=== All tools after full wiring ===
  agent.list_messages
  agent.send_message
  arithmetic.add
  arithmetic.divide
  arithmetic.multiply
  arithmetic.subtract
  code.execute
  file.list
  file.read
  file.write
  http.request
  memory.curate
  memory.read
  memory.search
  memory.write
  shell.execute
  web.search


In [46]:
# Use code.execute through the registry (just like an LLM would)
print("=== code.execute via registry ===")
result = registry.call("code.execute", {"code": "print(sum(range(101)))", "timeout": 5})
print(f"ok={result['ok']}, stdout='{result['data']['stdout'].strip()}'")

print()

# Use agent.send_message through the registry
print("=== agent.send_message via registry ===")
result = registry.call("agent.send_message", {
    "to": "analyzer",
    "content": {"task": "process data"},
    "channel": "tasks",
})
print(f"ok={result['ok']}")
print(f"message_id={result['data']['message_id'][:12]}...")
print(f"from={result['data']['from']}, to={result['data']['to']}, channel={result['data']['channel']}")

=== code.execute via registry ===
ok=True, stdout='5050'

=== agent.send_message via registry ===
ok=True
message_id=4657d855-ae6...
from=unknown, to=analyzer, channel=tasks


---
## 10. Summary: AWP Runtime Component Reference

| Component | Module | Purpose |
|-----------|--------|---------|
| **ToolRegistry** | `awp.runtime.tools` | Central hub for all tools (built-in, custom, dynamic). Provides `call()`, `get_definitions()`, `tool_names`. |
| **CodeExecutor** | `awp.runtime.code_executor` | Subprocess-based Python sandbox with timeout and output limits. |
| **create_executor()** | `awp.runtime.executor_factory` | Factory that creates the right executor from `SandboxConfig` (subprocess, docker, venv). |
| **MessageBus** | `awp.runtime.message_bus` | In-memory inter-agent messaging: direct, broadcast, channels. |
| **StatePersistence** | `awp.runtime.state_persistence` | JSON checkpoints per agent and final workflow state. |
| **CircuitBreaker** | `awp.runtime.security` | Stops calling failing services after N failures (closed/open/half_open). |
| **RateLimiter** | `awp.runtime.security` | Sliding window rate limiting per agent. |
| **AccessController** | `awp.runtime.security` | Tool-level access policies (deny lists per agent). |
| **SecurityContext** | `awp.runtime.security` | Composed security subsystems (CB + RL + AC). |
| **Tracer** | `awp.runtime.observability` | Span-based tracing with JSONL output. |
| **MetricsCollector** | `awp.runtime.observability` | Counters and histograms with JSON output. |
| **AuditTrail** | `awp.runtime.observability` | Hash-chain integrity audit log. |
| **ObservabilityContext** | `awp.runtime.observability` | Composed observability subsystems. |
| **DynamicToolFactory** | `awp.runtime.dynamic_tool_factory` | Runtime tool creation with AST validation and namespace enforcement. |

### Standard AWP Tool Result Format

Every tool returns:
```python
{"ok": bool, "status": int, "data": Any, "error": str | None}
```

### Key Integration Points

- `ToolRegistry.set_code_executor(executor)` -- registers `code.execute` tool
- `ToolRegistry.set_message_bus(bus)` -- registers `agent.send_message` and `agent.list_messages`
- `ToolRegistry.set_security_context(ctx)` -- enables per-agent access control
- `ToolRegistry.set_dynamic_tool_factory(factory)` -- enables runtime tool creation

All of these are wired automatically by the DAG Runner and Delegation Loop Runner
when executing workflows via `awp run`.